# YouTube Data Extraction for Taylor Swift Analytics

This notebook extracts data from Taylor Swift's official YouTube channel using the YouTube Data API v3.
The extracted data includes channel information, playlists, videos, and statistics.

## Import Required Libraries

In [ ]:
import os
import requests
import pandas as pd 
import json 
import datetime
import re
from dotenv import load_dotenv

# Configure pandas display options
pd.set_option('display.max_columns', None)

## Load Environment Variables and Configuration

In [ ]:
# Load environment variables from .env file
load_dotenv()
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found in environment variables")

In [ ]:
# Taylor Swift's official YouTube channel handle
CHANNEL_HANDLE = "@TaylorSwift"

## Extract Channel Information

First, we'll get basic channel information including the uploads playlist ID.

In [ ]:
def get_channel_info(handle, api_key):
    """
    Retrieve channel information using the channel handle.
    
    Args:
        handle (str): YouTube channel handle (e.g., @TaylorSwift)
        api_key (str): YouTube Data API key
    
    Returns:
        dict: Channel information from YouTube API
    """
    response = requests.get(
        "https://www.googleapis.com/youtube/v3/channels",
        params={
            "part": "snippet,statistics,contentDetails",
            "forHandle": handle,
            "key": api_key
        }
    )
    
    if response.status_code != 200:
        raise Exception(f"API request failed with status {response.status_code}")
    
    data = response.json()
    
    if not data.get("items"):
        raise Exception(f"No channel found for handle: {handle}")
    
    return data

# Get channel information
channel_response = get_channel_info(CHANNEL_HANDLE, YOUTUBE_API_KEY)
channel_item = channel_response["items"][0]

# Extract key information
channel_id = channel_item["id"]
uploads_playlist_id = channel_item["contentDetails"]["relatedPlaylists"]["uploads"]

print(f"Channel ID: {channel_id}")
print(f"Channel Title: {channel_item['snippet']['title']}")
print(f"Subscriber Count: {channel_item['statistics'].get('subscriberCount', 'Hidden')}")
print(f"Uploads Playlist ID: {uploads_playlist_id}")

## Process Channel Data for Export

In [ ]:
def process_channel_data(channel_response):
    """
    Process channel data into a structured format for export.
    
    Args:
        channel_response (dict): Raw channel response from YouTube API
    
    Returns:
        pd.DataFrame: Processed channel data
    """
    channels = []
    
    for item in channel_response.get("items", []):
        channel_data = {
            "channel_id": item["id"],
            "title": item["snippet"]["title"],
            "description": item["snippet"]["description"],
            "custom_url": item["snippet"].get("customUrl"),
            "published_at": item["snippet"]["publishedAt"],
            "country": item["snippet"].get("country"),
            "thumbnail_high": item["snippet"]["thumbnails"]["high"]["url"],
            "uploads_playlist": item["contentDetails"]["relatedPlaylists"]["uploads"],
            "subscriber_count": int(item["statistics"].get("subscriberCount", 0)),
            "view_count": int(item["statistics"].get("viewCount", 0)),
            "video_count": int(item["statistics"].get("videoCount", 0)),
        }
        channels.append(channel_data)
    
    return pd.DataFrame(channels)

# Process channel data
channels_df = process_channel_data(channel_response)
print("Channel data processed successfully")
channels_df.head()

## Extract Channel Playlists

In [ ]:
def get_channel_playlists(channel_id, api_key):
    """
    Retrieve all playlists for a given channel.
    
    Args:
        channel_id (str): YouTube channel ID
        api_key (str): YouTube Data API key
    
    Returns:
        list: List of playlist items from YouTube API
    """
    playlist_items = []
    page_token = None
    
    while True:
        params = {
            "part": "snippet,contentDetails",
            "channelId": channel_id,
            "maxResults": 50,
            "key": api_key
        }
        
        if page_token:
            params["pageToken"] = page_token
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/playlists",
            params=params
        )
        
        if response.status_code != 200:
            raise Exception(f"API request failed with status {response.status_code}")
        
        data = response.json()
        playlist_items.extend(data.get("items", []))
        
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    
    return playlist_items

# Get all playlists
playlist_items = get_channel_playlists(channel_id, YOUTUBE_API_KEY)
print(f"Found {len(playlist_items)} playlists")

In [ ]:
def process_playlist_data(playlist_items):
    """
    Process playlist data into a structured format for export.
    
    Args:
        playlist_items (list): Raw playlist items from YouTube API
    
    Returns:
        pd.DataFrame: Processed playlist data
    """
    playlists = []
    
    for item in playlist_items:
        playlist_data = {
            "playlist_id": item["id"],
            "title": item["snippet"]["title"],
            "description": item["snippet"]["description"],
            "published_at": item["snippet"]["publishedAt"],
            "channel_id": item["snippet"]["channelId"],
            "channel_title": item["snippet"]["channelTitle"],
            "item_count": item["contentDetails"]["itemCount"],
            "thumbnail_high": item["snippet"]["thumbnails"].get("high", {}).get("url", ""),
        }
        playlists.append(playlist_data)
    
    return pd.DataFrame(playlists)

# Process playlist data
playlists_df = process_playlist_data(playlist_items)
print("Playlist data processed successfully")
print(f"Shape: {playlists_df.shape}")
playlists_df.head()

## Extract Videos from Playlists

In [ ]:
def get_playlist_items(playlist_id, api_key):
    """
    Retrieve all items (videos) from a specific playlist.
    
    Args:
        playlist_id (str): YouTube playlist ID
        api_key (str): YouTube Data API key
    
    Returns:
        list: List of playlist items from YouTube API
    """
    items = []
    page_token = None
    
    while True:
        params = {
            "part": "snippet,contentDetails,status",
            "playlistId": playlist_id,
            "maxResults": 50,
            "key": api_key
        }
        
        if page_token:
            params["pageToken"] = page_token
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/playlistItems",
            params=params
        )
        
        if response.status_code != 200:
            print(f"Warning: Failed to fetch items for playlist {playlist_id}")
            break
        
        data = response.json()
        items.extend(data.get("items", []))
        
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    
    return items

# Get all playlist items
all_playlist_items = []

for playlist_id in playlists_df["playlist_id"]:
    items = get_playlist_items(playlist_id, YOUTUBE_API_KEY)
    
    # Add playlist_id to each item for reference
    for item in items:
        item["parent_playlist_id"] = playlist_id
    
    all_playlist_items.extend(items)

print(f"Total videos found across all playlists: {len(all_playlist_items)}")

In [ ]:
def process_playlist_items_data(playlist_items):
    """
    Process playlist items data into a structured format for export.
    
    Args:
        playlist_items (list): Raw playlist items from YouTube API
    
    Returns:
        pd.DataFrame: Processed playlist items data
    """
    if not playlist_items:
        return pd.DataFrame()
    
    # Normalize the JSON data
    df = pd.json_normalize(playlist_items, sep="__")
    
    # Rename columns for clarity
    column_mapping = {
        "id": "playlist_item_id",
        "snippet__title": "video_title",
        "snippet__description": "video_description",
        "snippet__publishedAt": "added_to_playlist_at",
        "contentDetails__videoPublishedAt": "video_published_at",
        "contentDetails__videoId": "video_id",
        "snippet__channelId": "channel_id",
        "snippet__channelTitle": "channel_title",
        "snippet__position": "position_in_playlist",
        "parent_playlist_id": "playlist_id",
        "status__privacyStatus": "privacy_status"
    }
    
    # Apply column mapping
    df = df.rename(columns=column_mapping)
    
    # Convert datetime columns
    datetime_columns = ["added_to_playlist_at", "video_published_at"]
    for col in datetime_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    return df

# Process playlist items data
playlist_items_df = process_playlist_items_data(all_playlist_items)
print("Playlist items data processed successfully")
print(f"Shape: {playlist_items_df.shape}")
playlist_items_df.head()

## Extract Video Statistics

In [ ]:
def get_video_statistics(video_ids, api_key, batch_size=50):
    """
    Retrieve detailed statistics for a list of video IDs.
    
    Args:
        video_ids (list): List of YouTube video IDs
        api_key (str): YouTube Data API key
        batch_size (int): Number of videos to process per API call
    
    Returns:
        list: List of video statistics from YouTube API
    """
    all_videos = []
    
    # Process videos in batches
    for i in range(0, len(video_ids), batch_size):
        batch_ids = video_ids[i:i + batch_size]
        ids_string = ",".join(batch_ids)
        
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/videos",
            params={
                "part": "snippet,contentDetails,statistics,liveStreamingDetails",
                "id": ids_string,
                "key": api_key
            },
            timeout=30
        )
        
        if response.status_code != 200:
            print(f"Warning: Failed to fetch video statistics for batch {i//batch_size + 1}")
            continue
        
        data = response.json()
        all_videos.extend(data.get("items", []))
    
    return all_videos

# Get unique video IDs
unique_video_ids = playlist_items_df["video_id"].dropna().unique().tolist()
print(f"Found {len(unique_video_ids)} unique videos")

# Get video statistics
video_statistics = get_video_statistics(unique_video_ids, YOUTUBE_API_KEY)
print(f"Retrieved statistics for {len(video_statistics)} videos")

In [ ]:
def process_video_statistics_data(video_statistics):
    """
    Process video statistics data into a structured format for export.
    
    Args:
        video_statistics (list): Raw video statistics from YouTube API
    
    Returns:
        pd.DataFrame: Processed video statistics data
    """
    if not video_statistics:
        return pd.DataFrame()
    
    # Normalize the JSON data
    df = pd.json_normalize(video_statistics, sep="__")
    
    # Column mapping for clarity
    column_mapping = {
        "id": "video_id",
        "snippet__title": "video_title",
        "snippet__description": "video_description",
        "snippet__publishedAt": "video_published_at",
        "contentDetails__duration": "duration_iso8601",
        "statistics__viewCount": "view_count",
        "statistics__likeCount": "like_count",
        "statistics__commentCount": "comment_count",
        "snippet__thumbnails__default__url": "thumbnail_default",
        "snippet__thumbnails__medium__url": "thumbnail_medium",
        "snippet__thumbnails__high__url": "thumbnail_high",
        "liveStreamingDetails__actualStartTime": "live_actual_start",
        "liveStreamingDetails__actualEndTime": "live_actual_end",
        "liveStreamingDetails__scheduledStartTime": "live_scheduled_start",
    }
    
    # Apply column mapping
    df = df.rename(columns=column_mapping)
    
    # Convert numeric columns
    numeric_columns = ["view_count", "like_count", "comment_count"]
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    
    # Convert datetime columns
    datetime_columns = ["video_published_at", "live_actual_start", "live_actual_end", "live_scheduled_start"]
    for col in datetime_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
    
    # Select relevant columns
    relevant_columns = [
        "video_id", "video_title", "video_description", "video_published_at",
        "duration_iso8601", "view_count", "like_count", "comment_count",
        "thumbnail_default", "thumbnail_medium", "thumbnail_high",
        "live_actual_start", "live_actual_end", "live_scheduled_start"
    ]
    
    # Keep only columns that exist in the dataframe
    existing_columns = [col for col in relevant_columns if col in df.columns]
    
    return df[existing_columns]

# Process video statistics data
video_stats_df = process_video_statistics_data(video_statistics)
print("Video statistics data processed successfully")
print(f"Shape: {video_stats_df.shape}")
video_stats_df.head()

## Export Data to CSV Files

In [ ]:
# Create data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Export all dataframes to CSV
export_files = {
    "channels.csv": channels_df,
    "playlists.csv": playlists_df,
    "playlist_items.csv": playlist_items_df,
    "video_stats.csv": video_stats_df
}

for filename, dataframe in export_files.items():
    filepath = f"data/{filename}"
    dataframe.to_csv(filepath, index=False)
    print(f"Exported {len(dataframe)} records to {filepath}")

print("\nData extraction completed successfully!")
print("CSV files are ready to be used as seeds in the dbt project.")

## Data Summary

The following data has been extracted and exported:

In [ ]:
print("=== DATA EXTRACTION SUMMARY ===")
print(f"Channel: {channels_df.iloc[0]['title'] if not channels_df.empty else 'N/A'}")
print(f"Total Playlists: {len(playlists_df)}")
print(f"Total Playlist Items: {len(playlist_items_df)}")
print(f"Unique Videos: {len(video_stats_df)}")
print(f"Total Views: {video_stats_df['view_count'].sum():,} (across all videos)")
print("\nFiles exported to data/ directory:")
for filename in export_files.keys():
    print(f"  - {filename}")